# Rebuild the Modelling Table

In [1]:

import numpy as np
import pandas as pd

RANDOM_STATE = 42
TARGET = "salary_in_usd"

df = pd.read_csv("../data/ds_salaries.csv")


In [ ]:
text_cols = [
    "experience_level",
    "employment_type",
    "job_title",
    "employee_residence",
    "company_location",
    "company_size"
]

# Strip leading and trailing spaces
for col in text_cols:
    df[col] = df[col].str.strip()

# Normalize text labels
for col in text_cols:
    if col != "job_title":
        df[col] = df[col].str.upper()

# Remove duplicate rows
df = df.drop_duplicates().reset_index(drop=True)

# Separate features and target
X = df.drop(columns=[TARGET])  
y = df[TARGET]

print(X.shape, y.shape)


(2584, 8) (2584,)


# Split the Data

In [ ]:

from sklearn.model_selection import train_test_split

# Split features and target into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=X["experience_level"]  
)

# Check split sizes
print(len(X_train), len(X_test))


2067 517


# Build the Preprocessing Pipeline

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

EXP_ORDER = ["EN", "MI", "SE", "EX"]  # junior -> executive

numeric_cols = ["work_year"]

onehot_low = [
    "employment_type",
    "company_size",
    "remote_ratio"
]

onehot_high = [
    "job_title",
    "employee_residence",
    "company_location"
]


In [6]:
def make_preprocessor(scale: bool) -> ColumnTransformer:
    exp_enc = OrdinalEncoder(categories=[EXP_ORDER])

    if scale:  # Linear models and KNN
        exp_enc = Pipeline([
            ("ord", exp_enc),
            ("sc", StandardScaler())
        ])

    return ColumnTransformer([
        ("exp", exp_enc, ["experience_level"]),

        (
            "num",
            StandardScaler() if scale else "passthrough",
            numeric_cols
        ),

        (
            "cat_low",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            onehot_low
        ),

        # Rare categories (< 1% of TRAINING rows) are pooled automatically
        (
            "cat_high",
            OneHotEncoder(
                handle_unknown="infrequent_if_exist",
                min_frequency=0.01,
                sparse_output=False
            ),
            onehot_high
        )
    ])